# Clean `emr_cycle`

A first-pass cleaning of the `emr_cycle` table (9,276 rows, ~200 columns) before analysis.

What this notebook does, in order:
1. **Profile** every column (how full it is, how many distinct values, its type)
2. **Drop** columns that are completely empty
3. **Review** columns that hold only a single value
4. **Parse** the date columns (currently stored as text)
5. **Tidy** column types
6. **Save** a cleaned copy + keep an audit note of what changed

The same `profile()` function works on the other `emr_*` tables too, so you can reuse this pattern.

> **Before running:** finish the venv setup in this repo and install the packages — open a terminal and run `pip install pandas pyarrow`. The `requirements.txt` alongside this notebook lists everything.
>
> **Important:** keep your raw patient data in a `data/` folder that is git-ignored. Don't commit EMR exports to GitHub, even a private repo.

## Setup

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

import import_ipynb

import helper_functions

<string>:48: SyntaxWarning: invalid escape sequence '\s'


In [2]:
# --- point this at your raw export ---
PATH = Path("../data/emr_cycle.csv")

# loading data
df = pd.read_csv(PATH)

C:\Users\tamar.schaap\AppData\Local\Temp\ipykernel_47708\891529149.py:5: DtypeWarning: Columns (0: deletedon, 1: plan_sperm, 2: eggdonorid, 3: checklist_done, 4: incubator, 5: fin_status, 6: pat_instructions, 7: pregnancystart, 8: accessionid, 9: icsi_witness_other, 10: billing_status) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(PATH)


In [3]:
#setting display options to show all columns and full width
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.width', None)        # Allow full width

## 1. Profile every column

One row per column so you can see at a glance what's worth keeping.

In [4]:
# Already know there are many all NULL columns, so dropping those first
print(df.shape)
df = df.dropna(axis=1, how="all")
print(df.shape)

(9775, 220)
(9775, 101)


In [5]:
# now profiling the dataframes to get a better understanding of the data and dropping unninformative columns

df_prof = helper_functions.profile(df)
df = helper_functions.drop_empty_and_constant_columns(df, df_prof)
print(df_prof)

# patid = Registry System ID in nAble Patient PM
# sart_id = SART Cycle ID in nAble (this is the only column that is directly searchable in nAble)
# planned_eggsource = Account number in nAble Patient PM
# partner_id -> Registry System ID in nAble Patient PM for the partner
# planned_spermsource -> Account number in nAble Patient PM for the partner

Dropping 22 columns
                    non_null  nulls  distinct    dtype  pct_null  \
id                      9775      0      9775      str       0.0   
patid                   9775      0      3746    int64       0.0   
addedby                 9775      0        43    int64       0.0   
addedon                 9775      0      8537      str       0.0   
status                  9775      0         7      str       0.0   
...                      ...    ...       ...      ...       ...   
plan_sperm                 1   9774         1      str     100.0   
accessionid                1   9774         1      str     100.0   
icsi_witness_other         1   9774         1      str     100.0   
billing_status             1   9774         1      str     100.0   
cycleinsurance             1   9774         1  float64     100.0   

                    all_values_same  
id                            False  
patid                         False  
addedby                       False  
addedon    

## 2. Cleaning Data

pulling details from cycle_name

In [ ]:
# parse cycle_name
parsed = df.apply(helper_functions.parse_cycle_row, axis=1, result_type='expand')
df = pd.concat([df, parsed], axis=1)

# Add the broad family column
df['protocol_family_broad'] = df.apply(helper_functions.derive_protocol_family_broad, axis=1)

# Quick QC
print(df['classification_status'].value_counts())


classification_status
OK                             9260
OK (no protocol expected)       234
REVIEW: no protocol_family      170
REVIEW: UUID/regex mismatch     111
Name: count, dtype: int64


## 2. Keeping subset of columns to organize into separate dfs

Only looking to keep columns that inform response type. Filtering beyond the NA and single value columns

In [32]:
# first renaming cycleid
df = df.rename(columns={"id": "cycleid"})

### df for FET cycles

In [33]:
# ----- Saving a df for FET cycles

# confirmed_transfers = rows where cycle_name contains "FET"
# keep columns: patid, partnerid, sart_id, status, closed, cycletype, cycle_name, cyclestart, cycleend, event_labels, plan_treatment, hist_monthsattempting, main_note, height, weight, bmi, hist_smoker, hist_pat_surg_sterile

confirmed_transfers = df[
    df["cycle_name"].astype(str).str.contains("FET", case=False, na=False)
][[
    "cycleid", "patid", "partnerid", "sart_id", "status",
    "closed", "cycletype", "cycle_name", "cyclestart", 
    "cycleend", "event_labels", "plan_treatment", 
    "hist_monthsattempting", "main_note", "height", "weight",
    "bmi", "hist_smoker", "hist_pat_surg_sterile", "doctor"
]]

print(confirmed_transfers.shape)
#display(confirmed_transfers.head())

(3559, 20)


### df for egg retrievals

In [44]:
confirmed_retrievals = df[
    df["cycle_name"].astype(str).str.contains(
        "IVF|Oocyte Freeze", case=False, na=False, regex=True
    )
    & ~df["cycle_name"].astype(str).str.contains(
        "Third Party|Donor|Donation|converted|->", case=False, na=False, regex=True
    )
][[
    "cycleid", "patid", "partnerid", "sart_id", "status",
    "closed", "cycletype", "cycle_name", "cycle_type",
    "clinic", "protocol_family", "protocol_detail",
    "protocol_family_broad", "protocol_family_uuid", 
    "protocol_family_regex", "adjuvants", "cyclestart", 
    "cycleend", "end_reason", "event_labels", "plan_treatment", 
    "hist_monthsattempting", "main_note", "height", "weight",
    "bmi", "hist_smoker", "hist_pat_surg_sterile", "doctor",
    "art_reason",
]]

In [ ]:
#simply processed df

In [45]:
# create a list of columns that I want to keep
keep_cols = [
    "cycleid", "sart_id", "doctor", "patid", "partnerid", "barcode", "chloe_id", "height", "weight", 
    "bmi", "addedby", "status", "closed", "cycletype", "schedulestart", "cyclestart", "cycleend", 
    "scheduleend", "event_labels", "doctor", "plan_treatment", "main_note", "art_reason", 
    "cycle_name", "cyclecoordinator", "icsi_tech", "icsi_witness", "insemination_tech", 
    "hist_smoker", "hist_pat_surg_sterile", "hist_monthsattempting", "manual_culturestart",
    "planned_start_date", "planned_retrieval_date", "orig_scheduleend", "edd", "calc_basedate", 
    "calc_type", "last_prospective_report_id", "last_final_report_id", "pgt_reason", "pgt_type", 
    "last_final_report_date", "eggsource_eligibility", "spermsource_eligibility", 
    "planned_eggsource", "planned_spermsource", "end_reason", "icsi_reason", "clinic", "protocol_family", 
    "protocol_detail", "protocol_family_broad", "protocol_family_uuid", "protocol_family_regex", "adjuvants", 
]
# filter the dataframe to keep only those columns
df = df[keep_cols]
print(f"\nAfter filtering to {len(keep_cols)} columns: {df.shape[1]} columns")


After filtering to 56 columns: 56 columns


## 3. Saving as a CSV

In [ ]:
# save as csv
confirmed_transfers.to_csv("../data/transfer_cycles.csv", index=False)
confirmed_retrievals.to_csv("../data/retrieval_cycles.csv", index=False)
df.to_csv("../data/emr_cycle_processed.csv", index=False)